<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/Exp04_MultiSource.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exp4 — Multi-Source Pre-Train + Fine-Tune

**Zweck:** Untersuchen, ob ein Modell, das auf mehreren Source-Städten gemeinsam trainiert wurde,
robusteren Transfer liefert als Single-Source — insbesondere für Target-Städte, bei denen
kein einzelner Source gut funktioniert (z.B. Mannheim).

**Design:**
- Multi-Source-Modell: FC trainiert auf kombinierten Trainingsdaten mehrerer Source-Städte
- Gemeinsamer Scaler über alle Source-Städte (damit das Modell verschiedene Demand-Skalen sieht)
- Zero-Shot: Multi-Source-Modell direkt auf Target anwenden
- Fine-Tune: Multi-Source-Modell auf Budget-Fenster der Target-Stadt weitertrainieren
- Evaluation: Strict + Fair (identisch zu Exp3)

**Vergleich mit:**
- Exp2 (bester Single-Source Zero-Shot)
- Exp3 (bester Single-Source Fine-Tune)
- Exp01 (Oracle + Scarcity)

**Kernfrage:** Reduziert Multi-Source die Varianz zwischen Stadtpaaren?

In [1]:
# ══════════════════════════════════════════════════════════════
# 0a — Colab Setup
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════════════
# 0b — Imports
# ══════════════════════════════════════════════════════════════
import os, math, json, random, warnings, time, re, glob, copy, pickle
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import poisson
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve,
    classification_report
)
from numpy.lib.stride_tricks import sliding_window_view

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
print(f"TF {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")


TF 2.19.0, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ══════════════════════════════════════════════════════════════
# 0c — Config (Exp2: Strict Zero-Shot, Multiple Source→Target Pairs)
# ══════════════════════════════════════════════════════════════
DATA_BASE    = '/content/drive/MyDrive/BA_Colab/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

RUN_NAME    = 'exp02_zero_shot_strict_pairs'
RESULTS_DIR = f'{DATA_BASE}/{RUN_NAME}'
MODELS_DIR  = f'{RESULTS_DIR}/models'

# Quelle der bereits trainierten Exp01-Oracle-Modelle
SOURCE_RUN_NAME   = 'exp01_oracle_scarcity'
SOURCE_MODELS_DIR = f'{DATA_BASE}/{SOURCE_RUN_NAME}/models'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Paths
geo_path           = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
station_names_path = f'{DATA_BASE}/station_names/station_names.parquet'

CITY_DEMAND = {
    "bochum":        f"{CLEANED_BASE}/demand/Bochum/demand_cleaned.parquet",
    "braunschweig":  f"{CLEANED_BASE}/demand/Braunschweig/demand_cleaned.parquet",
    "dortmund":      f"{CLEANED_BASE}/demand/Dortmund/demand_cleaned.parquet",
    "duisburg":      f"{CLEANED_BASE}/demand/Duisburg/demand_cleaned.parquet",
    "essen":         f"{CLEANED_BASE}/demand/Essen/demand_cleaned.parquet",
    "freiburg":      f"{CLEANED_BASE}/demand/Freiburg/demand_cleaned.parquet",
    "gießen":        f"{CLEANED_BASE}/demand/Gießen/demand_cleaned.parquet",
    "heidelberg":    f"{CLEANED_BASE}/demand/Heidelberg/demand_cleaned.parquet",
    "kaiserslautern":f"{CLEANED_BASE}/demand/Kaiserslautern/demand_cleaned.parquet",
    "kassel":        f"{CLEANED_BASE}/demand/Kassel/demand_cleaned.parquet",
    "leverkusen":    f"{CLEANED_BASE}/demand/Leverkusen/demand_cleaned.parquet",
    "ludwigshafen":  f"{CLEANED_BASE}/demand/Ludwigshafen/demand_cleaned.parquet",
    "mannheim":      f"{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet",
    "marburg":       f"{CLEANED_BASE}/demand/Marburg/demand_cleaned.parquet",
    "Offenburg":     f"{CLEANED_BASE}/demand/Offenburg/demand_cleaned.parquet",
    "wiesbaden":     f"{CLEANED_BASE}/demand/Wiesbaden/demand_cleaned.parquet",
    "winsen (Luhe)": f"{CLEANED_BASE}/demand/Winsen (Luhe)/demand_cleaned.parquet",
}

# Vorgegebene Exp2-Paare
TRANSFER_PAIRS = {
    "mannheim": ["freiburg", "heidelberg", "wiesbaden"],
    "ludwigshafen": ["essen", "kaiserslautern"],
    "bochum": ["kassel", "marburg"],
}

PAIR_ROWS = [
    {"target_city": target_city, "source_city": source_city}
    for target_city, source_list in TRANSFER_PAIRS.items()
    for source_city in source_list
]

print("Geplante Zero-Shot-Paare:")
for target_city, source_list in TRANSFER_PAIRS.items():
    print(f"  Target {target_city}: {', '.join(source_list)}")

@dataclass
class V17bConfig:
    aggregation_minutes: int = 60
    train_ratio: float = 0.67
    val_ratio: float = 0.82
    min_events_per_day: float = 3.0
    rolling_days: int = 56
    min_context_obs: int = 20

    ae_window_size: int = 24
    ae_latent_dim: int = 32
    ae_layers: int = 2
    ae_dropout: float = 0.10
    ae_epochs: int = 50
    ae_batch_size: int = 2048
    ae_lr: float = 1e-3
    ae_early_stop: int = 8
    eval_batch_size: int = 2048

    low_demand_q: float = 0.33
    high_demand_q: float = 0.67

    require_contiguous: bool = True
    use_bidirectional: bool = False

    ae_features: List[str] = field(default_factory=lambda: [
        "n_lends", "n_returns",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "month_sin", "month_cos", "is_weekend"
    ])
    ae_score_features: List[str] = field(default_factory=lambda: ["n_lends", "n_returns"])

    injection_rate: float = 0.015
    injection_seed: int = 42

cfg = V17bConfig()

ae_features    = list(cfg.ae_features)
score_features = list(cfg.ae_score_features)
context_steps  = cfg.ae_window_size - 1

print(f"Feature-Set: {len(ae_features)} Features → {ae_features}")
print(f"Score-Features: {score_features}")
print(f"Context Steps: {context_steps}")
print(f"Results: {RESULTS_DIR}")
print(f"Source models: {SOURCE_MODELS_DIR}")


Geplante Zero-Shot-Paare:
  Target mannheim: freiburg, heidelberg, wiesbaden
  Target ludwigshafen: essen, kaiserslautern
  Target bochum: kassel, marburg
Feature-Set: 9 Features → ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
Score-Features: ['n_lends', 'n_returns']
Context Steps: 23
Results: /content/drive/MyDrive/BA_Colab/data/exp02_zero_shot_strict_pairs
Source models: /content/drive/MyDrive/BA_Colab/data/exp01_oracle_scarcity/models


---
## Shared Functions (aus V22b/V17b 1:1 übernommen)


In [4]:
# ══════════════════════════════════════════════════════════════
# 1 — Aggregation
# ══════════════════════════════════════════════════════════════
def aggregate_station_level(df: pd.DataFrame, minutes: int = 60,
                            add_relative: bool = False) -> pd.DataFrame:
    out = df.copy()
    freq = f"{minutes}min"
    out["time_bin"] = out["timestamp"].dt.floor(freq)

    agg = (
        out.groupby(
            ["station_id", "station_name_id", "station_name", "location_id", "time_bin"],
            as_index=False
        )
        .agg({
            "n_lends": "sum",
            "n_returns": "sum",
            "latitude": "first",
            "longitude": "first"
        })
        .rename(columns={"time_bin": "hour_ts"})
    )

    agg["total_demand"] = agg["n_lends"] + agg["n_returns"]
    agg["net_flow"] = agg["n_returns"] - agg["n_lends"]
    agg["abs_net_flow"] = agg["net_flow"].abs()

    agg["hour"] = agg["hour_ts"].dt.hour
    agg["dow"] = agg["hour_ts"].dt.dayofweek
    agg["month"] = agg["hour_ts"].dt.month
    agg["is_weekend"] = agg["dow"].isin([5, 6]).astype(int)

    agg["hour_sin"] = np.sin(2 * np.pi * agg["hour"] / 24)
    agg["hour_cos"] = np.cos(2 * np.pi * agg["hour"] / 24)
    agg["dow_sin"]  = np.sin(2 * np.pi * agg["dow"] / 7)
    agg["dow_cos"]  = np.cos(2 * np.pi * agg["dow"] / 7)
    agg["month_sin"] = np.sin(2 * np.pi * (agg["month"] - 1) / 12)
    agg["month_cos"] = np.cos(2 * np.pi * (agg["month"] - 1) / 12)

    agg = agg.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    return agg


In [5]:
# ══════════════════════════════════════════════════════════════
# 2 — Gap-Fill
# ══════════════════════════════════════════════════════════════
def fill_missing_time_bins(x: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    freq = f"{minutes}min"
    parts = []
    x = x.copy().sort_values(["station_id", "hour_ts"])
    x = (
        x.groupby(["station_id", "hour_ts"], as_index=False)
         .agg({
             "station_name_id": "first", "station_name": "first",
             "location_id": "first", "latitude": "first",
             "longitude": "first", "n_lends": "sum", "n_returns": "sum",
         })
    )
    key_cols = ["station_id", "station_name_id", "station_name",
                "location_id", "latitude", "longitude"]

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").copy()
        full_idx = pd.date_range(g["hour_ts"].min(), g["hour_ts"].max(), freq=freq)
        g = g.set_index("hour_ts").reindex(full_idx)
        g.index.name = "hour_ts"
        g["n_lends"]   = g["n_lends"].fillna(0).astype(int)
        g["n_returns"] = g["n_returns"].fillna(0).astype(int)
        for col in key_cols:
            g[col] = g[col].ffill().bfill()
        parts.append(g.reset_index())

    result = pd.concat(parts, ignore_index=True)
    result["total_demand"] = result["n_lends"] + result["n_returns"]
    result["net_flow"]     = result["n_returns"] - result["n_lends"]
    result["abs_net_flow"] = result["net_flow"].abs()

    result["hour"] = result["hour_ts"].dt.hour
    result["dow"]  = result["hour_ts"].dt.dayofweek
    result["month"] = result["hour_ts"].dt.month
    result["is_weekend"] = result["dow"].isin([5, 6]).astype(int)

    result["hour_sin"]  = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"]  = np.cos(2 * np.pi * result["hour"] / 24)
    result["dow_sin"]   = np.sin(2 * np.pi * result["dow"] / 7)
    result["dow_cos"]   = np.cos(2 * np.pi * result["dow"] / 7)
    result["month_sin"] = np.sin(2 * np.pi * (result["month"] - 1) / 12)
    result["month_cos"] = np.cos(2 * np.pi * (result["month"] - 1) / 12)

    return result.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)


In [6]:
# ══════════════════════════════════════════════════════════════
# 3 — Station-Filter, Demand-Regime, Train/Val/Test Split
# ══════════════════════════════════════════════════════════════
def prepare_stations_and_splits(df: pd.DataFrame, cfg, city_name=""):
    n_days = (df["hour_ts"].max() - df["hour_ts"].min()).days + 1
    min_events = int(n_days * cfg.min_events_per_day)

    station_volume = df.groupby("station_id")["total_demand"].sum()
    active_ids = station_volume[station_volume >= min_events].index.tolist()
    df = df[df["station_id"].isin(active_ids)].copy()

    station_profile = (
        df.groupby(["station_id", "station_name"], as_index=False)
          .agg(
              avg_total_demand_h=("total_demand", "mean"),
              avg_lends_h=("n_lends", "mean"),
              avg_returns_h=("n_returns", "mean"),
              latitude=("latitude", "first"),
              longitude=("longitude", "first")
          )
    )
    q1 = station_profile["avg_total_demand_h"].quantile(cfg.low_demand_q)
    q2 = station_profile["avg_total_demand_h"].quantile(cfg.high_demand_q)
    station_profile["demand_regime"] = station_profile["avg_total_demand_h"].apply(
        lambda x: "low" if x <= q1 else ("mid" if x <= q2 else "high")
    )
    df = df.merge(
        station_profile[["station_id", "demand_regime", "avg_total_demand_h"]],
        on="station_id", how="left"
    )

    t_min = df["hour_ts"].min()
    t_max = df["hour_ts"].max()
    total_h = (t_max - t_min).total_seconds() / 3600

    train_end = t_min + pd.Timedelta(hours=int(total_h * cfg.train_ratio))
    val_end   = t_min + pd.Timedelta(hours=int(total_h * cfg.val_ratio))

    cn = f" ({city_name})" if city_name else ""
    print(f"  Aktive Stationen{cn}: {df['station_id'].nunique()}")
    print(f"  Regime: {station_profile['demand_regime'].value_counts().to_dict()}")
    print(f"  Zeitraum: {t_min.date()} bis {t_max.date()} ({n_days} Tage)")
    print(f"  Split: Train < {train_end.date()}, Val < {val_end.date()}, Test ab {val_end.date()}")

    return df, station_profile, train_end, val_end


In [7]:
# ══════════════════════════════════════════════════════════════
# 4 — Rolling Context z-Scores + Poisson + ECDF + Labels
# ══════════════════════════════════════════════════════════════
TARGETS = ["n_lends", "n_returns", "net_flow", "total_demand"]
COUNT_TARGETS = ["n_lends", "n_returns", "total_demand"]

def add_context_keys(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    x["ctx_hour"] = x["hour_ts"].dt.hour
    x["ctx_dow"]  = x["hour_ts"].dt.dayofweek
    return x

def rolling_context_scores_vectorized(
    full_df: pd.DataFrame, target: str,
    rolling_days: int = 56, min_obs: int = 20
) -> pd.DataFrame:
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    mu_col    = f"{target}_mu_ctx_roll"
    sd_col    = f"{target}_sd_ctx_roll"
    score_col = f"{target}_z_ctx_roll"

    n_slots = max(rolling_days // 7, 4)
    main_window = n_slots
    main_minp   = min(min_obs, main_window)

    grouped = x.groupby(["station_id", "ctx_hour", "ctx_dow"])[target]
    rolling_mean = grouped.transform(
        lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).mean()
    )
    rolling_std = grouped.transform(
        lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).std(ddof=0)
    )

    fb1_window = n_slots * 2
    fb1_minp   = min(min_obs, fb1_window)
    grouped_sh = x.groupby(["station_id", "ctx_hour"])[target]
    fb1_mean = grouped_sh.transform(
        lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).mean()
    )
    fb1_std = grouped_sh.transform(
        lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).std(ddof=0)
    )

    fb2_window = n_slots * 4
    fb2_minp   = min(min_obs, fb2_window)
    grouped_s = x.groupby(["station_id"])[target]
    fb2_mean = grouped_s.transform(
        lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).mean()
    )
    fb2_std = grouped_s.transform(
        lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).std(ddof=0)
    )

    mu = rolling_mean.copy()
    sd = rolling_std.copy()
    mask1 = mu.isna()
    mu = mu.where(~mask1, fb1_mean)
    sd = sd.where(~mask1, fb1_std)
    mask2 = mu.isna()
    mu = mu.where(~mask2, fb2_mean)
    sd = sd.where(~mask2, fb2_std)
    sd = sd.clip(lower=1e-6)
    z = (x[target] - mu) / sd

    x[mu_col]    = mu
    x[sd_col]    = sd
    x[score_col] = z
    return x


def add_rolling_poisson_scores_vectorized(
    full_df: pd.DataFrame, target: str,
    rolling_days: int = 56, min_obs: int = 20
) -> pd.DataFrame:
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    lam_col       = f"{target}_lambda_ctx_roll"
    score_col     = f"{target}_poisson_upper_score"
    score_low_col = f"{target}_poisson_lower_score"
    mu_col        = f"{target}_mu_ctx_roll"

    if mu_col not in x.columns:
        raise ValueError(f"{mu_col} muss zuerst berechnet werden")

    x[lam_col] = x[mu_col].clip(lower=1e-6)
    vals = x[target].values
    lams = x[lam_col].values

    with np.errstate(divide='ignore', invalid='ignore'):
        tail_p = poisson.sf(vals.astype(float) - 1, lams.astype(float))
        score  = -np.log10(np.clip(tail_p, 1e-12, None))
        lower_p   = poisson.cdf(vals.astype(float), lams.astype(float))
        score_low = -np.log10(np.clip(lower_p, 1e-12, None))

    mask_nan = np.isnan(lams)
    score[mask_nan]     = np.nan
    score_low[mask_nan] = np.nan

    x[score_col]     = score
    x[score_low_col] = score_low
    return x


def percentile_score(train_vals: np.ndarray, values: np.ndarray) -> np.ndarray:
    train_vals = np.asarray(train_vals, dtype=float)
    values     = np.asarray(values, dtype=float)
    train_vals = train_vals[np.isfinite(train_vals)]
    if len(train_vals) == 0:
        return np.full(len(values), np.nan, dtype=float)
    train_sorted = np.sort(train_vals)
    ranks = np.searchsorted(train_sorted, values, side="right")
    return ranks / len(train_sorted)


def run_statistical_pipeline(df, cfg, train_end):
    df = add_context_keys(df)
    for t in TARGETS:
        df = rolling_context_scores_vectorized(df, t, cfg.rolling_days, cfg.min_context_obs)
    for t in COUNT_TARGETS:
        df = add_rolling_poisson_scores_vectorized(df, t, cfg.rolling_days, cfg.min_context_obs)

    train_mask = df["hour_ts"] < train_end
    score_cols = [c for c in df.columns if c.endswith("_z_ctx_roll")]

    for c in score_cols:
        train_vals = df.loc[train_mask, c].dropna().values
        df[f"{c}_ecdf"] = percentile_score(train_vals, df[c].values)

    def label_row(row):
        for t in TARGETS:
            z_col = f"{t}_z_ctx_roll"
            if z_col in row.index and pd.notna(row[z_col]):
                if row[z_col] >= 3.0:
                    return "anomal_high"
                if row[z_col] <= -3.0:
                    return "anomal_low"
        return "normal"

    df["label_eval"] = df.apply(label_row, axis=1)
    return df


In [8]:
# ══════════════════════════════════════════════════════════════
# 5 — Sequenzbuilder (Window-Level Labels)
# ══════════════════════════════════════════════════════════════
def make_sequences_with_window_labels(
    x: pd.DataFrame, feature_cols: List[str], window_size: int,
    synth_label_col: str = "synth_label",
    synth_type_col: str = "synth_type",
    require_contiguous: bool = True,
    agg_minutes: int = 60
) -> Tuple[np.ndarray, pd.DataFrame]:
    X_parts, meta_parts = [], []
    expected_ns = pd.Timedelta(minutes=agg_minutes).value

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").reset_index(drop=True)
        n = len(g)
        if n < window_size:
            continue

        vals = g[feature_cols].to_numpy(dtype=np.float32)
        win  = sliding_window_view(vals, window_shape=window_size, axis=0)
        win  = np.moveaxis(win, -1, 1)

        valid_mask = np.ones(n - window_size + 1, dtype=bool)

        if require_contiguous:
            ts_int = pd.to_datetime(g["hour_ts"]).astype("int64").to_numpy()
            diffs  = np.diff(ts_int)
            step_ok = (diffs == expected_ns).astype(np.int8)
            if window_size > 1:
                ok_sums = np.convolve(step_ok, np.ones(window_size-1, dtype=np.int32), mode="valid")
                valid_mask = (ok_sums == (window_size - 1))

        if not valid_mask.any():
            continue

        end_indices = np.arange(window_size - 1, n)[valid_mask]
        X_parts.append(win[valid_mask])

        meta_chunk = g.iloc[end_indices].copy()

        synth_arr = g[synth_label_col].to_numpy()
        type_arr  = g[synth_type_col].to_numpy()

        window_labels, window_types, window_counts = [], [], []
        for end_idx in end_indices:
            start_idx = end_idx - window_size + 1
            window_synth = synth_arr[start_idx:end_idx + 1]
            window_type  = type_arr[start_idx:end_idx + 1]

            has_synth = int(window_synth.max())
            n_synth   = int(window_synth.sum())

            if has_synth:
                synth_positions = np.where(window_synth == 1)[0]
                wtype = window_type[synth_positions[-1]]
            else:
                wtype = "none"

            window_labels.append(has_synth)
            window_types.append(wtype)
            window_counts.append(n_synth)

        meta_chunk["window_synth_label"] = window_labels
        meta_chunk["window_synth_type"]  = window_types
        meta_chunk["n_synth_in_window"]  = window_counts
        meta_parts.append(meta_chunk)

    if X_parts:
        X    = np.concatenate(X_parts, axis=0)
        meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)
    else:
        X    = np.empty((0, window_size, len(feature_cols)), dtype=np.float32)
        meta = pd.DataFrame()

    return X, meta


In [9]:
# ══════════════════════════════════════════════════════════════
# 6 — Model Builders (AE + Forecaster)
# ══════════════════════════════════════════════════════════════
def build_lstm_autoencoder(
    n_input_features: int,
    window_size: int,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
    bidirectional: bool = False,
    n_output_features: Optional[int] = None,
) -> keras.Model:
    n_output_features = n_output_features or n_input_features
    inputs = keras.Input(shape=(window_size, n_input_features), name="encoder_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        lstm = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"encoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"encoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    latent = x
    x = layers.RepeatVector(window_size, name="repeat_vector")(latent)
    for i in range(n_layers):
        lstm = layers.LSTM(
            latent_dim, return_sequences=True,
            dropout=dropout, name=f"decoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"decoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    outputs = layers.TimeDistributed(
        layers.Dense(n_output_features), name="output_dense"
    )(x)
    return keras.Model(inputs, outputs, name="lstm_autoencoder")


def build_lstm_forecaster(
    n_input_features: int,
    n_output_features: int,
    context_steps: int = 23,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
) -> keras.Model:
    inputs = keras.Input(shape=(context_steps, n_input_features), name="forecast_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        x = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"forecast_lstm_{i+1}"
        )(x)
    outputs = layers.Dense(n_output_features, name="forecast_output")(x)
    return keras.Model(inputs, outputs, name="lstm_forecaster")


def train_model_generic(model, X_train, Y_train, X_val, Y_val,
                        epochs, lr, batch_size, early_stop, verbose=1):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss="mse"
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=early_stop,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, verbose=1
        )
    ]
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )
    return model, history


In [10]:
def train_model_fixed_epochs(model, X_train, Y_train, epochs, lr, batch_size, verbose=1):
    """Training mit fixer Epochenzahl. Kein Val-Set, kein Early Stopping. CosineDecay LR."""
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=lr,
        decay_steps=epochs * max(1, len(X_train) // batch_size),
        alpha=1e-6
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0), loss="mse")
    history = model.fit(X_train, Y_train, epochs=epochs, batch_size=batch_size, verbose=verbose)
    return model, {k: [float(v) for v in vals] for k, vals in history.history.items()}


In [11]:
# ══════════════════════════════════════════════════════════════
# 7 — Scoring Helpers (AE + Forecaster)
# ══════════════════════════════════════════════════════════════
def predict_in_batches(model, X, batch_size=2048):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(model.predict(X[i:i + batch_size], verbose=0))
    return np.concatenate(preds, axis=0) if preds else np.empty((0, 0, 0), dtype=np.float32)


def compute_reconstruction_scores(X, X_hat, input_features, score_features,
                                  mode="last_step_mean"):
    score_idx = [input_features.index(c) for c in score_features]
    err = (X - X_hat) ** 2
    err_score = err[:, :, score_idx]
    if mode == "last_step_mean":
        return err_score[:, -1, :].mean(axis=1)
    elif mode == "window_mean":
        return err_score.mean(axis=(1, 2))
    elif mode == "max_over_features":
        return err_score[:, -1, :].max(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")


def compute_forecast_scores(model, X_input, X_actual_last_step, score_features,
                            ae_features, mode="directional"):
    X_pred = predict_in_batches(model, X_input, batch_size=2048)
    score_idx = [ae_features.index(f) for f in score_features]
    X_actual = X_actual_last_step[:, score_idx]
    err = (X_pred - X_actual) ** 2
    if mode == "mean":
        return err.mean(axis=1)
    elif mode == "max":
        return err.max(axis=1)
    elif mode == "directional":
        abs_err = np.abs(X_pred - X_actual)
        denom = np.maximum(np.maximum(np.abs(X_actual), np.abs(X_pred)), 1e-6)
        rel_err = abs_err / denom
        return 0.5 * abs_err.mean(axis=1) + 0.5 * rel_err.mean(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")


In [12]:
# ══════════════════════════════════════════════════════════════
# 8 — Z-Normalisierung (Hour + Hour×Station)
# ══════════════════════════════════════════════════════════════
def znorm_score_by_hour(meta_df, raw_score_col, val_start, test_start,
                        new_col="score_znorm_hour"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    mu_per_hour = {}
    std_per_hour = {}
    for h in range(24):
        vals = meta.loc[val_normal & (hour_col == h), raw_score_col].dropna()
        mu_per_hour[h]  = vals.mean() if len(vals) > 0 else 0.0
        std_per_hour[h] = vals.std()  if len(vals) > 1 else 1.0

    hours = hour_col.values
    raw   = meta[raw_score_col].values
    znorm = np.zeros_like(raw)
    for i in range(len(raw)):
        h = hours[i]
        znorm[i] = (raw[i] - mu_per_hour[h]) / max(std_per_hour[h], 1e-8)
    meta[new_col] = znorm
    return meta, mu_per_hour, std_per_hour


def znorm_score_by_hour_station(meta_df, raw_score_col, val_start, test_start,
                                new_col="score_znorm_hour_station"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    meta["_hour_tmp"] = hour_col

    stats = (
        meta[val_normal]
        .groupby(["station_id", "_hour_tmp"])[raw_score_col]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    global_stats = (
        meta[val_normal]
        .groupby("_hour_tmp")[raw_score_col]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "g_mean", "std": "g_std"})
    )
    stats = stats.merge(global_stats, on="_hour_tmp", how="left")
    stats["use_mean"] = np.where(stats["count"] >= 10, stats["mean"], stats["g_mean"])
    stats["use_std"]  = np.where(stats["count"] >= 10, stats["std"],  stats["g_std"])
    stats["use_std"]  = stats["use_std"].clip(lower=1e-8)

    meta = meta.merge(
        stats[["station_id", "_hour_tmp", "use_mean", "use_std"]],
        on=["station_id", "_hour_tmp"], how="left"
    )
    meta["use_mean"] = meta["use_mean"].fillna(0.0)
    meta["use_std"]  = meta["use_std"].fillna(1.0).clip(lower=1e-8)
    meta[new_col] = (meta[raw_score_col] - meta["use_mean"]) / meta["use_std"]
    meta = meta.drop(columns=["_hour_tmp", "use_mean", "use_std"])
    return meta


In [13]:
# ══════════════════════════════════════════════════════════════
# 9 — Synthetic Injection (V17b: Spike Only)
# ══════════════════════════════════════════════════════════════
V17B_INJECTION_PROBS = {
    "point_spike": 1.0
}
V17B_COLLECTIVE_BLOCK_LEN = (2, 4)

def inject_synthetic_anomalies_v17b(
    df: pd.DataFrame,
    inject_start: pd.Timestamp,
    injection_rate: float = 0.015,
    seed: int = 42,
    verbose: bool = True
) -> pd.DataFrame:
    probs = list(V17B_INJECTION_PROBS.values())
    type_names = list(V17B_INJECTION_PROBS.keys())
    block_min, block_max = V17B_COLLECTIVE_BLOCK_LEN

    rng = np.random.RandomState(seed)
    out = df.copy()

    out["original_n_lends"]   = out["n_lends"].copy()
    out["original_n_returns"] = out["n_returns"].copy()
    out["synth_label"] = 0
    out["synth_type"]  = "none"

    inject_mask = out["hour_ts"] >= inject_start
    inject_idx  = out[inject_mask].index.to_numpy()

    if len(inject_idx) == 0:
        print("  WARNUNG: Keine Daten ab inject_start!")
        return out

    n_inject = max(1, int(len(inject_idx) * injection_rate))
    inject_df = out.loc[inject_idx]
    has_demand = inject_df[inject_df["total_demand"] > 0].index.to_numpy()

    if len(has_demand) < n_inject:
        n_inject = len(has_demand)

    chosen_idx = rng.choice(has_demand, size=n_inject, replace=False)
    types = rng.choice(type_names, size=n_inject, p=probs)

    injected_indices = set()
    requested_counts = {t: 0 for t in type_names}
    injected_counts  = {t: 0 for t in type_names}

    for idx, anom_type in zip(chosen_idx, types):
        requested_counts[anom_type] += 1
        if idx in injected_indices:
            continue

        row = out.loc[idx]

        if anom_type == "point_spike":
            factor = rng.uniform(2.0, 4.0)
            out.loc[idx, "n_lends"]   = int(row["n_lends"] * factor) + rng.randint(1, 4)
            out.loc[idx, "n_returns"] = int(row["n_returns"] * factor) + rng.randint(1, 4)
            out.loc[idx, "synth_label"] = 1
            out.loc[idx, "synth_type"]  = "point_spike"
            injected_indices.add(idx)
            injected_counts["point_spike"] += 1

    out.loc[list(injected_indices), "total_demand"] = (
        out.loc[list(injected_indices), "n_lends"] +
        out.loc[list(injected_indices), "n_returns"]
    )

    if verbose:
        print(f"  Injection: requested={sum(requested_counts.values())}, "
              f"injected={sum(injected_counts.values())}")
        for t in type_names:
            print(f"    {t}: req={requested_counts[t]}, inj={injected_counts[t]}")

    return out


---
## City Data Pipeline + Evaluation (Exp0: ohne Wetter-Merge)


In [14]:
# ══════════════════════════════════════════════════════════════
# 10 — City Data Pipeline (Exp0: ohne Wetter-Merge)
# ══════════════════════════════════════════════════════════════
# ANPASSUNG gegenüber V22b: Wetter-Parameter und Wetter-Merge entfernt,
# da Exp0 nur das Baseline-Feature-Set (demand + zyklisch) verwendet.
# Alle anderen Schritte sind identisch.

def prepare_city_data(demand_path: str, geo_path: str, station_names_path: str,
                      cfg, city_name: str):
    print(f"\n{'='*70}")
    print(f"  DATEN-PIPELINE: {city_name}")
    print(f"{'='*70}")

    demand = pd.read_parquet(demand_path)
    geo    = pd.read_parquet(geo_path)
    snames = pd.read_parquet(station_names_path)
    snames = snames.rename(columns={'id': 'station_name_id', 'name': 'station_name'})

    demand = demand.merge(snames[['station_name_id', 'station_name']], on='station_name_id', how='left')
    demand = demand.merge(geo[['location_id', 'latitude', 'longitude']], on='location_id', how='left')
    demand['timestamp'] = pd.to_datetime(demand['timestamp'], utc=True)

    print(f"  Roh: {len(demand):,} Zeilen, {demand['station_id'].nunique()} Stationen")
    print(f"  Zeitraum: {demand['timestamp'].min().date()} bis {demand['timestamp'].max().date()}")

    print("\n  [1] Aggregation...")
    df = aggregate_station_level(demand, minutes=cfg.aggregation_minutes)
    print(f"      Shape: {df.shape}")

    print("  [2] Gap-Fill...")
    n_before = len(df)
    df = fill_missing_time_bins(df, minutes=cfg.aggregation_minutes)
    print(f"      {n_before:,} -> {len(df):,} (+{len(df)-n_before:,})")

    print("  [3] Stationen, Regime, Splits...")
    df, station_profile, train_end, val_end = prepare_stations_and_splits(df, cfg, city_name)

    print("  [4] Statistische Scoring-Pipeline...")
    df = run_statistical_pipeline(df, cfg, train_end)

    return df, station_profile, train_end, val_end


In [15]:
# ══════════════════════════════════════════════════════════════
# 11 — V17b Unified Evaluation
# ══════════════════════════════════════════════════════════════
ANOM_TYPES = ["point_spike"]
REGIMES    = ["high", "mid", "low"]

def evaluate_v17b(
    meta: pd.DataFrame,
    score_col: str,
    experiment_name: str,
    val_start: pd.Timestamp,
    test_start: pd.Timestamp,
    verbose: bool = True
) -> Dict:
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test")
    )

    val_m  = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()

    if len(val_m) == 0 or len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine VAL/TEST-Daten!")
        return {}

    results = {"experiment": experiment_name, "n_val": len(val_m), "n_test": len(test_m)}

    n_test_total = len(test_m)
    n_test_anom  = (test_m["synth_label"] == 1).sum()
    results["anomaly_rate_total"] = n_test_anom / n_test_total
    results["n_test_anom"] = int(n_test_anom)

    for atype in ANOM_TYPES:
        n_t = (test_m["synth_type"] == atype).sum()
        results[f"rate_{atype}"] = n_t / n_test_total
        results[f"n_{atype}"] = int(n_t)

    # Threshold via VAL
    y_val = val_m["synth_label"].astype(int).values
    s_val = val_m[score_col].values
    threshold = None
    if len(np.unique(y_val)) > 1:
        prec_v, rec_v, thr_v = precision_recall_curve(y_val, s_val)
        results["val_pr_auc"] = average_precision_score(y_val, s_val)
        if len(thr_v) > 0:
            f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
            best_idx = np.argmax(f1_v)
            threshold = float(thr_v[best_idx])
            results["val_best_f1"] = float(f1_v[best_idx])
            results["val_threshold"] = threshold

    if threshold is None:
        val_norm = val_m[val_m["synth_label"] == 0][score_col]
        threshold = float(val_norm.quantile(0.99)) if len(val_norm) > 0 else float(test_m[score_col].quantile(0.99))
        results["val_threshold"] = threshold
        results["threshold_source"] = "fallback_99pct"

    # TEST Metriken
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= threshold).astype(int)

    if len(np.unique(y_test)) > 1:
        results["test_pr_auc"]  = average_precision_score(y_test, s_test)
        results["test_roc_auc"] = roc_auc_score(y_test, s_test)
        results["test_f1"]      = f1_score(y_test, p_test, zero_division=0)
        results["test_prec"]    = precision_score(y_test, p_test, zero_division=0)
        results["test_recall"]  = recall_score(y_test, p_test, zero_division=0)

    for atype in ANOM_TYPES:
        type_mask = test_m["synth_type"] == atype
        normal_mask = test_m["synth_label"] == 0
        sub = test_m[type_mask | normal_mask].copy()
        n_anom = int(type_mask.sum())
        if n_anom > 0 and len(sub) > 0:
            y_t = (sub["synth_type"] == atype).astype(int).values
            y_s = sub[score_col].values
            if len(set(y_t)) > 1:
                results[f"pr_{atype}"] = average_precision_score(y_t, y_s)
            detected = (test_m.loc[type_mask, score_col] >= threshold).sum()
            results[f"dr_{atype}"] = detected / n_anom
        else:
            results[f"pr_{atype}"] = None
            results[f"dr_{atype}"] = None

    # Regime × Type
    regime_rows = []
    for atype in ANOM_TYPES:
        for regime in REGIMES:
            rm = test_m["demand_regime"] == regime
            tm = test_m["synth_type"] == atype
            nm = test_m["synth_label"] == 0
            n_a = int((rm & tm).sum())
            n_n = int((rm & nm).sum())
            pr, f1_val, dr = None, None, None
            if n_a > 0 and n_n > 0:
                sub = test_m[(rm & tm) | (rm & nm)]
                y_t = (sub["synth_type"] == atype).astype(int).values
                y_s = sub[score_col].values
                pr = average_precision_score(y_t, y_s) if len(set(y_t)) > 1 else None
                p_sub = (y_s >= threshold).astype(int)
                f1_val = f1_score(y_t, p_sub, zero_division=0)
                detected = (test_m[rm & tm][score_col] >= threshold).sum()
                dr = detected / n_a
            regime_rows.append({
                "type": atype, "regime": regime,
                "n_anom": n_a, "pr_auc": pr, "f1": f1_val, "dr": dr
            })
    results["regime_detail"] = pd.DataFrame(regime_rows)

    if verbose:
        print(f"\n  [{experiment_name}]")
        print(f"    TEST PR-AUC={results.get('test_pr_auc', 'N/A'):.4f}, "
              f"F1={results.get('test_f1', 'N/A'):.4f}, "
              f"Prec={results.get('test_prec', 'N/A'):.4f}, "
              f"Recall={results.get('test_recall', 'N/A'):.4f}")

    return results


---
## Exp0 — City Discovery + Oracle-Loop


In [16]:
# ══════════════════════════════════════════════════════════════
# MC-1 — City Discovery
# ══════════════════════════════════════════════════════════════
import re
from pathlib import Path

def city_to_key(city_name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", city_name.lower()).strip("_")

def discover_city_demand_paths(cleaned_base: str):
    demand_root = Path(cleaned_base) / "demand"
    city_map = {}
    if not demand_root.exists():
        print(f"WARNUNG: demand_root nicht gefunden: {demand_root}")
        return city_map
    for city_dir in sorted(demand_root.iterdir()):
        if not city_dir.is_dir():
            continue
        p = city_dir / "demand_cleaned.parquet"
        if p.exists():
            city_map[city_dir.name] = str(p)
    return city_map

DISCOVERED_CITY_DEMAND = discover_city_demand_paths(CLEANED_BASE)
CITY_DEMAND_ALL = {**DISCOVERED_CITY_DEMAND, **CITY_DEMAND}

print(f"Gefundene Städte: {len(CITY_DEMAND_ALL)}")

# ── Zielstädte (hier anpassen) ──────────────────────────────
TARGET_CITIES = [
    "bochum",
    "braunschweig",
    "dortmund",
    "duisburg",
    "essen",
    "freiburg",
    "gießen",
    "heidelberg",
    "kaiserslautern",
    "kassel",
    "leverkusen",
    "ludwigshafen",
    "mannheim",
    "marburg",
    "offenburg",
    "wiesbaden",
    "winsen (Luhe)",
]

TARGET_CITIES = [c for c in TARGET_CITIES if c in CITY_DEMAND_ALL]
print("\nTarget Cities:", TARGET_CITIES)


Gefundene Städte: 33

Target Cities: ['bochum', 'braunschweig', 'dortmund', 'duisburg', 'essen', 'freiburg', 'gießen', 'heidelberg', 'kaiserslautern', 'kassel', 'leverkusen', 'ludwigshafen', 'mannheim', 'marburg', 'wiesbaden', 'winsen (Luhe)']


In [17]:
# ══════════════════════════════════════════════════════════════
# MC-2 — Speicherung + Flatten Helpers
# ══════════════════════════════════════════════════════════════
def save_window_level_scores(meta, city_name, model_name, out_dir,
                             score_cols=None, extra_cols=None):
    score_cols = score_cols or []
    extra_cols = extra_cols or []
    base_cols = [
        "station_id", "hour_ts", "split_eval", "label_eval",
        "synth_label", "synth_type", "demand_regime"
    ]
    cols = [c for c in base_cols + score_cols + extra_cols if c in meta.columns]
    out = meta[cols].copy()
    out["city"] = city_name
    out["city_key"] = city_to_key(city_name)
    out["model"] = model_name

    fpath = f"{out_dir}/{city_to_key(city_name)}__{model_name}__window_scores.parquet"
    out.to_parquet(fpath, index=False)
    print(f"  Gespeichert: {fpath} | rows={len(out):,}")
    return fpath

def flatten_result_dict(d: dict, city_name: str, model_name: str):
    row = {}
    for k, v in d.items():
        if isinstance(v, pd.DataFrame):
            continue
        row[k] = v
    row["city"] = city_name
    row["city_key"] = city_to_key(city_name)
    row["model"] = model_name
    return row

def add_station_demand_weight(meta: pd.DataFrame):
    meta = meta.copy()
    demand_col = next((c for c in ["original_n_lends", "n_lends"] if c in meta.columns), None)
    if demand_col is None:
        stat = meta.groupby("station_id").size().rename("station_activity_raw").reset_index()
    else:
        stat = (
            meta.groupby("station_id")[demand_col]
                .mean()
                .rename("station_activity_raw")
                .reset_index()
        )
    stat["station_activity_log1p"] = np.log1p(stat["station_activity_raw"].clip(lower=0))
    q33 = stat["station_activity_raw"].quantile(0.33)
    q67 = stat["station_activity_raw"].quantile(0.67)
    stat["station_activity_group"] = np.where(
        stat["station_activity_raw"] <= q33, "low",
        np.where(stat["station_activity_raw"] >= q67, "high", "medium")
    )
    meta = meta.merge(stat, on="station_id", how="left")
    return meta


In [18]:
# ══════════════════════════════════════════════════════════════
# MC-3a — Sequenz-Erstellung
# ══════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler

def build_sequences_for_city(
    df_clean, df_inj, train_end, val_end, city_name, scaler_ext=None
):
    """
    Generische Sequenz-Erstellung pro Stadt.
    Liest ae_features und score_features aus dem globalen Scope.
    """
    global ae_features, score_features, context_steps

    df_c = df_clean.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    df_inj_s = df_inj.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)

    df_c["hour_ts"] = pd.to_datetime(df_c["hour_ts"], utc=True)
    df_inj_s["hour_ts"] = pd.to_datetime(df_inj_s["hour_ts"], utc=True)
    train_end = pd.to_datetime(train_end, utc=True)
    val_end   = pd.to_datetime(val_end, utc=True)

    # Prüfe ob alle Features vorhanden
    missing = [f for f in ae_features if f not in df_c.columns]
    if missing:
        print(f"  [{city_name}] WARNUNG: Fehlende Features: {missing}")
        return None

    df_c = df_c.dropna(subset=ae_features)
    df_inj_s = df_inj_s.dropna(subset=ae_features)

    train_mask_sc = df_c["hour_ts"] < train_end
    if scaler_ext is None:
        scaler = StandardScaler()
        scaler.fit(df_c.loc[train_mask_sc, ae_features])
        print(f"  [{city_name}] Scaler gefittet auf {train_mask_sc.sum():,} Train-Zeilen ({len(ae_features)} Features)")
    else:
        scaler = scaler_ext

    df_inj_scaled = df_inj_s.copy()
    df_inj_scaled.loc[:, ae_features] = scaler.transform(df_inj_s[ae_features])

    meta_cols_to_keep = [
        "synth_label", "synth_type", "original_n_lends", "original_n_returns",
        "demand_regime", "label_eval", "station_id", "hour_ts", "n_lends", "n_returns",
    ]
    for col in meta_cols_to_keep:
        if col in df_inj_s.columns:
            df_inj_scaled[col] = df_inj_s[col].values

    X_all, meta_all = make_sequences_with_window_labels(
        df_inj_scaled,
        feature_cols=ae_features,
        window_size=cfg.ae_window_size,
        synth_label_col="synth_label",
        synth_type_col="synth_type",
        require_contiguous=cfg.require_contiguous,
        agg_minutes=cfg.aggregation_minutes
    )

    if "hour_ts" in meta_all.columns:
        meta_all["hour_ts"] = pd.to_datetime(meta_all["hour_ts"], utc=True)

    meta_all["split_eval"] = np.where(
        meta_all["hour_ts"] < train_end, "train",
        np.where(meta_all["hour_ts"] < val_end, "val", "test")
    )

    print(f"  [{city_name}] Sequenzen: {X_all.shape}")
    print(f"  [{city_name}] Split: {meta_all['split_eval'].value_counts().to_dict()}")

    train_normal = (meta_all["split_eval"] == "train") & (meta_all["label_eval"] == "normal")
    val_normal = (
        (meta_all["split_eval"] == "val") &
        (meta_all["synth_label"] == 0) &
        (meta_all["label_eval"] == "normal")
    )
    X_train = X_all[train_normal.values]
    X_val_c = X_all[val_normal.values]

    print(f"  [{city_name}] X_train (normal): {X_train.shape}, X_val (normal): {X_val_c.shape}")

    return {
        "X_all": X_all, "meta_all": meta_all, "scaler": scaler,
        "X_train": X_train, "X_val_clean": X_val_c,
        "train_normal_mask": train_normal, "val_normal_mask": val_normal,
        "train_end": train_end, "val_end": val_end,
    }


In [26]:
# ══════════════════════════════════════════════════════════════
# 0c — Config (Exp4: Multi-Source Pre-Train + Fine-Tune)
# ══════════════════════════════════════════════════════════════
DATA_BASE    = '/content/drive/MyDrive/BA_Colab/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

RUN_NAME    = 'exp04_multi_source'
RESULTS_DIR = f'{DATA_BASE}/{RUN_NAME}'
MODELS_DIR  = f'{RESULTS_DIR}/models'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

geo_path           = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
station_names_path = f'{DATA_BASE}/station_names/station_names.parquet'

CITY_DEMAND = {
    "bochum":        f"{CLEANED_BASE}/demand/Bochum/demand_cleaned.parquet",
    "braunschweig":  f"{CLEANED_BASE}/demand/Braunschweig/demand_cleaned.parquet",
    "dortmund":      f"{CLEANED_BASE}/demand/Dortmund/demand_cleaned.parquet",
    "duisburg":      f"{CLEANED_BASE}/demand/Duisburg/demand_cleaned.parquet",
    "essen":         f"{CLEANED_BASE}/demand/Essen/demand_cleaned.parquet",
    "freiburg":      f"{CLEANED_BASE}/demand/Freiburg/demand_cleaned.parquet",
    "gießen":        f"{CLEANED_BASE}/demand/Gießen/demand_cleaned.parquet",
    "heidelberg":    f"{CLEANED_BASE}/demand/Heidelberg/demand_cleaned.parquet",
    "kaiserslautern":f"{CLEANED_BASE}/demand/Kaiserslautern/demand_cleaned.parquet",
    "kassel":        f"{CLEANED_BASE}/demand/Kassel/demand_cleaned.parquet",
    "leverkusen":    f"{CLEANED_BASE}/demand/Leverkusen/demand_cleaned.parquet",
    "ludwigshafen":  f"{CLEANED_BASE}/demand/Ludwigshafen/demand_cleaned.parquet",
    "mannheim":      f"{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet",
    "marburg":       f"{CLEANED_BASE}/demand/Marburg/demand_cleaned.parquet",
    "Offenburg":     f"{CLEANED_BASE}/demand/Offenburg/demand_cleaned.parquet",
    "wiesbaden":     f"{CLEANED_BASE}/demand/Wiesbaden/demand_cleaned.parquet",
    "winsen (Luhe)": f"{CLEANED_BASE}/demand/Winsen (Luhe)/demand_cleaned.parquet",
}

# ══════════════════════════════════════════════════════════════
# Multi-Source Konfiguration
# ══════════════════════════════════════════════════════════════
# Pro Target: welche Sources werden kombiniert?
# Die Sources sind die gleichen wie in Exp2/3 (Single-Source),
# aber jetzt als gemeinsames Trainingsset.

RUN_NAME = 'exp04_multi_source'


#MULTI_SOURCE_CONFIGS = {
    #"mannheim":      {"sources": ["freiburg", "heidelberg", "wiesbaden"]},
    #"ludwigshafen":  {"sources": ["essen", "kaiserslautern"]},
    #"bochum":        {"sources": ["kassel", "marburg"]},
    #"gießen":        {"sources": ["offenburg", "leverkusen", "marburg"]},
    #"dortmund":      {"sources": ["braunschweig", "marburg", "heidelberg"]},
    #"wiesbaden":     {"sources": ["ludwigshafen", "kaiserslautern", "heidelberg"]},
#}

MULTI_SOURCE_CONFIGS = {
    "gießen":        {"sources": ["Offenburg", "leverkusen", "marburg"]},
    "dortmund":      {"sources": ["braunschweig", "marburg", "heidelberg"]},
    "wiesbaden":     {"sources": ["ludwigshafen", "kaiserslautern", "heidelberg"]},
}



# Fine-Tune Budgets (0 = Zero-Shot only)
FT_BUDGETS_DAYS = [0, 7, 14, 30]

# Training-Parameter
MS_EPOCHS  = 50      # mehr Epochen, da mehr Daten
MS_LR      = 1e-3
FT_EPOCHS  = 20
FT_LR      = 3e-4

@dataclass
class V17bConfig:
    aggregation_minutes: int = 60
    train_ratio: float = 0.67
    val_ratio: float = 0.82
    min_events_per_day: float = 3.0
    rolling_days: int = 56
    min_context_obs: int = 20

    ae_window_size: int = 24
    ae_latent_dim: int = 32
    ae_layers: int = 2
    ae_dropout: float = 0.10
    ae_epochs: int = 50
    ae_batch_size: int = 2048
    ae_lr: float = 1e-3
    ae_early_stop: int = 8
    eval_batch_size: int = 2048

    low_demand_q: float = 0.33
    high_demand_q: float = 0.67

    require_contiguous: bool = True
    use_bidirectional: bool = False

    ae_features: List[str] = field(default_factory=lambda: [
        "n_lends", "n_returns",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "month_sin", "month_cos", "is_weekend"
    ])
    ae_score_features: List[str] = field(default_factory=lambda: ["n_lends", "n_returns"])

    injection_rate: float = 0.015
    injection_seed: int = 42

cfg = V17bConfig()

ae_features    = list(cfg.ae_features)
score_features = list(cfg.ae_score_features)
context_steps  = cfg.ae_window_size - 1

print("Multi-Source Konfiguration:")
for target, conf in MULTI_SOURCE_CONFIGS.items():
    print(f"  Target {target}: Sources={conf['sources']}")
print(f"FT Budgets: {FT_BUDGETS_DAYS}")
print(f"MS Training: {MS_EPOCHS} Ep, LR={MS_LR}")
print(f"FT: {FT_EPOCHS} Ep, LR={FT_LR}")
print(f"Results: {RESULTS_DIR}")


Multi-Source Konfiguration:
  Target gießen: Sources=['Offenburg', 'leverkusen', 'marburg']
  Target dortmund: Sources=['braunschweig', 'marburg', 'heidelberg']
  Target wiesbaden: Sources=['ludwigshafen', 'kaiserslautern', 'heidelberg']
FT Budgets: [0, 7, 14, 30]
MS Training: 50 Ep, LR=0.001
FT: 20 Ep, LR=0.0003
Results: /content/drive/MyDrive/BA_Colab/data/exp04_multi_source


---
## Exp4 — Multi-Source Helpers

1. **`build_multi_source_model`**: Lädt Daten aller Sources, fittet gemeinsamen Scaler,
   baut Sequenzen, trainiert FC auf kombiniertem Datensatz
2. **`run_multi_source_transfer`**: Wendet Multi-Source-Modell auf Target an (Zero-Shot oder Fine-Tune)

In [27]:
# ══════════════════════════════════════════════════════════════
# Exp4-1 — Multi-Source Helpers
# ══════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler

def compute_best_f1_threshold(y_true, scores):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    if len(np.unique(y_true)) <= 1:
        thr = float(np.quantile(scores, 0.99)) if len(scores) else 0.0
        return thr, np.nan, "fallback_99pct"
    prec, rec, thr = precision_recall_curve(y_true, scores)
    if len(thr) == 0:
        return float(np.quantile(scores, 0.99)), np.nan, "fallback_99pct"
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f1))
    return float(thr[best_idx]), float(f1[best_idx]), "multi_source_val_best_f1"

def evaluate_fixed_threshold(meta, score_col, threshold, experiment_name,
                              val_start, test_start, verbose=True):
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test"))
    test_m = meta[meta["split_eval"] == "test"].copy()
    results = {"experiment": experiment_name, "val_threshold": float(threshold), "n_test": len(test_m)}
    if len(test_m) == 0:
        return results
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= threshold).astype(int)
    results["n_test_anom"] = int((y_test == 1).sum())
    if len(np.unique(y_test)) > 1:
        results["test_pr_auc_raw"] = float(average_precision_score(y_test, s_test))
    results["test_f1"] = float(f1_score(y_test, p_test, zero_division=0))
    results["test_prec"] = float(precision_score(y_test, p_test, zero_division=0))
    results["test_recall"] = float(recall_score(y_test, p_test, zero_division=0))
    results["pred_pos_rate_test"] = float(p_test.mean())
    if verbose:
        print(f"  [{experiment_name}]")
        print(f"    TEST raw PR-AUC={results.get('test_pr_auc_raw', 'N/A'):.4f}, "
              f"F1={results['test_f1']:.4f}, Prec={results['test_prec']:.4f}, "
              f"Recall={results['test_recall']:.4f}")
    return results


def build_multi_source_model(source_cities, cfg, models_dir, ms_epochs, ms_lr):
    """
    Trainiert ein FC-Modell auf den kombinierten Trainingsdaten mehrerer Source-Städte.

    Schritte:
    1. Lade und verarbeite jede Source-Stadt separat
    2. Fitte einen gemeinsamen StandardScaler über alle Source-Train-Daten
    3. Skaliere alle Daten mit dem gemeinsamen Scaler
    4. Baue Sequenzen für jede Stadt und konkateniere sie
    5. Trainiere FC auf dem kombinierten Datensatz (mit Early Stopping auf kombinierten Val)

    Returns: dict mit Modell, Scaler, Threshold, Metadaten
    """
    global ae_features, score_features, context_steps

    source_key = "__".join(sorted([city_to_key(c) for c in source_cities]))
    model_path = f"{models_dir}/ms_{source_key}_forecaster.keras"
    scaler_path = f"{models_dir}/ms_{source_key}_scaler.pkl"

    print("=" * 90)
    print(f"[MULTI-SOURCE] Training auf: {', '.join(source_cities)}")
    print("=" * 90)

    # ── Schritt 1+2: Alle Sources laden, gemeinsamen Scaler fitten ──
    all_train_raw = []   # Rohdaten für Scaler-Fitting
    city_artifacts = {}  # Pro Stadt: df_clean, df_inj, train_end, val_end

    for ci, src_city in enumerate(source_cities):
        print(f"\n  [{ci+1}/{len(source_cities)}] Lade {src_city}...")
        df_src, profile_src, train_end_src, val_end_src = prepare_city_data(
            CITY_DEMAND_ALL[src_city], geo_path, station_names_path, cfg, src_city)
        df_src_inj = inject_synthetic_anomalies_v17b(
            df_src, inject_start=train_end_src,
            injection_rate=cfg.injection_rate, seed=cfg.injection_seed + ci, verbose=True)

        # Rohe Train-Daten für gemeinsamen Scaler sammeln
        train_mask = df_src["hour_ts"] < train_end_src
        train_feats = df_src.loc[train_mask, ae_features].dropna()
        all_train_raw.append(train_feats)
        print(f"    {src_city}: {len(train_feats):,} Train-Zeilen")

        city_artifacts[src_city] = {
            "df_clean": df_src, "df_inj": df_src_inj,
            "train_end": train_end_src, "val_end": val_end_src,
        }

    # Gemeinsamer Scaler
    combined_train_raw = pd.concat(all_train_raw, ignore_index=True)
    ms_scaler = StandardScaler()
    ms_scaler.fit(combined_train_raw[ae_features])
    print(f"\n  [MULTI-SOURCE] Gemeinsamer Scaler gefittet auf {len(combined_train_raw):,} Zeilen")

    # Scaler speichern
    import pickle
    with open(scaler_path, "wb") as f:
        pickle.dump(ms_scaler, f)

    del all_train_raw, combined_train_raw
    gc.collect()

    # ── Schritt 3+4: Sequenzen mit gemeinsamem Scaler bauen ─
    all_X_train = []
    all_X_val = []
    all_meta_val = []

    for src_city, art in city_artifacts.items():
        print(f"\n  Sequenzen für {src_city}...")
        city_data = build_sequences_for_city(
            df_clean=art["df_clean"], df_inj=art["df_inj"],
            train_end=art["train_end"], val_end=art["val_end"],
            city_name=src_city, scaler_ext=ms_scaler)

        if city_data is None:
            print(f"    {src_city}: ÜBERSPRUNGEN")
            continue

        all_X_train.append(city_data["X_train"])
        all_X_val.append(city_data["X_val_clean"])

        # Val-Meta für Threshold-Berechnung
        val_mask = city_data["meta_all"]["split_eval"] == "val"
        val_meta = city_data["meta_all"][val_mask].copy()
        val_meta["_source_city"] = src_city
        all_meta_val.append((city_data["X_all"][val_mask.values], val_meta, city_data))

        del art["df_clean"], art["df_inj"]
        gc.collect()

    X_train_combined = np.concatenate(all_X_train, axis=0)
    X_val_combined = np.concatenate(all_X_val, axis=0)
    print(f"\n  [MULTI-SOURCE] Kombiniert: X_train={X_train_combined.shape}, X_val={X_val_combined.shape}")

    # ── Schritt 5: Modell trainieren ─────────────────────────
    score_idx = [ae_features.index(f) for f in score_features]

    if os.path.exists(model_path):
        ms_model = keras.models.load_model(model_path, compile=False)
        print(f"  [MULTI-SOURCE] Modell geladen: {model_path}")
    else:
        print(f"  [MULTI-SOURCE] Training FC ({ms_epochs} Ep, {len(X_train_combined)} Seq)...")
        ms_model = build_lstm_forecaster(
            n_input_features=len(ae_features),
            n_output_features=len(score_features),
            context_steps=context_steps,
            latent_dim=cfg.ae_latent_dim, n_layers=2, dropout=cfg.ae_dropout)

        X_fc_train = X_train_combined[:, :context_steps, :]
        Y_fc_train = X_train_combined[:, -1, :][:, score_idx]
        X_fc_val   = X_val_combined[:, :context_steps, :]
        Y_fc_val   = X_val_combined[:, -1, :][:, score_idx]

        ms_model, _ = train_model_generic(
            ms_model, X_fc_train, Y_fc_train, X_fc_val, Y_fc_val,
            epochs=ms_epochs, lr=ms_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop, verbose=0)

        ms_model.save(model_path)
        print(f"  [MULTI-SOURCE] Modell gespeichert: {model_path}")

    # ── Threshold: Über ALLE Source-Val-Sets ──────────────────
    all_val_y = []
    all_val_s = []
    for X_val_city, meta_val_city, city_data_ref in all_meta_val:
        X_fc_in = city_data_ref["X_all"][:, :context_steps, :]
        X_act   = city_data_ref["X_all"][:, -1, :]
        all_scores = compute_forecast_scores(
            ms_model, X_fc_in, X_act,
            score_features, ae_features, mode="directional")
        val_mask = city_data_ref["meta_all"]["split_eval"] == "val"
        y_v = city_data_ref["meta_all"].loc[val_mask, "synth_label"].astype(int).values
        s_v = all_scores[val_mask.values]
        all_val_y.append(y_v)
        all_val_s.append(s_v)

    y_val_combined = np.concatenate(all_val_y)
    s_val_combined = np.concatenate(all_val_s)
    ms_threshold, ms_best_f1, thr_method = compute_best_f1_threshold(y_val_combined, s_val_combined)
    print(f"  [MULTI-SOURCE] Threshold ({thr_method}) = {ms_threshold:.6f}")
    if not np.isnan(ms_best_f1):
        print(f"  [MULTI-SOURCE] Combined Val F1 = {ms_best_f1:.4f}")

    # Cleanup
    del X_train_combined, X_val_combined, all_X_train, all_X_val, all_meta_val
    gc.collect()

    return {
        "ms_model": ms_model,
        "ms_scaler": ms_scaler,
        "ms_threshold_raw": ms_threshold,
        "source_cities": source_cities,
        "source_key": source_key,
        "model_path": model_path,
    }


def run_multi_source_transfer(ms_artifacts, target_city, cfg, budget_days,
                               ft_epochs, ft_lr, models_dir, seed_offset=0):
    """
    Multi-Source Transfer: Zero-Shot (budget_days=0) oder Fine-Tune.
    """
    global ae_features, score_features, context_steps

    source_key = ms_artifacts["source_key"]
    sources_str = "+".join(ms_artifacts["source_cities"])
    pair_id = f"[{sources_str}]\u2192{target_city}"
    mode = "ZeroShot" if budget_days == 0 else f"FT_{budget_days}d"
    print(f"\n[MULTI-SOURCE {mode}] {pair_id}")

    # ── Target-Daten mit Multi-Source-Scaler ─────────────────
    df_tgt, profile_tgt, train_end_tgt, val_end_tgt = prepare_city_data(
        CITY_DEMAND_ALL[target_city], geo_path, station_names_path, cfg, target_city)
    df_tgt_inj = inject_synthetic_anomalies_v17b(
        df_tgt, inject_start=train_end_tgt,
        injection_rate=cfg.injection_rate, seed=cfg.injection_seed + seed_offset, verbose=True)
    target_data = build_sequences_for_city(
        df_clean=df_tgt, df_inj=df_tgt_inj,
        train_end=train_end_tgt, val_end=val_end_tgt,
        city_name=target_city, scaler_ext=ms_artifacts["ms_scaler"])
    if target_data is None:
        return None, None
    target_data["meta_all"] = add_station_demand_weight(target_data["meta_all"])

    X_all = target_data["X_all"]
    meta  = target_data["meta_all"].copy()
    score_idx = [ae_features.index(f) for f in score_features]

    # ── Zero-Shot oder Fine-Tune ─────────────────────────────
    if budget_days == 0:
        # Zero-Shot: Modell direkt anwenden
        eval_model = ms_artifacts["ms_model"]
        n_ft = 0
        actual_days = 0
        train_time = 0
    else:
        # Fine-Tune: Budget-Fenster
        train_start = meta.loc[meta["split_eval"] == "train", "hour_ts"].min()
        budget_end  = train_start + pd.Timedelta(days=budget_days)
        budget_end  = min(budget_end, target_data["train_end"])
        budget_mask = (
            (meta["split_eval"] == "train") &
            (meta["label_eval"] == "normal") &
            (meta["hour_ts"] >= train_start) &
            (meta["hour_ts"] < budget_end))
        X_budget = X_all[budget_mask.values]
        n_ft = len(X_budget)
        actual_days = (budget_end - train_start).total_seconds() / 86400

        if n_ft < 10:
            print(f"  [{target_city}] Nur {n_ft} Seq \u2014 \u00fcbersprungen")
            return None, None

        city_key_tgt = city_to_key(target_city)
        ft_tag = f"ms_{source_key}_to_{city_key_tgt}_ft_{budget_days}d"
        ft_path = f"{models_dir}/{ft_tag}_forecaster.keras"
        batch_size_used = min(cfg.ae_batch_size, max(32, n_ft // 4))

        t0 = time.time()
        if os.path.exists(ft_path):
            eval_model = keras.models.load_model(ft_path, compile=False)
            print(f"  [{target_city}] FT-Modell geladen: {ft_path}")
        else:
            print(f"  [{target_city}] Fine-Tune ({ft_epochs} Ep, {n_ft} Seq)...")
            eval_model = build_lstm_forecaster(
                n_input_features=len(ae_features),
                n_output_features=len(score_features),
                context_steps=context_steps,
                latent_dim=cfg.ae_latent_dim, n_layers=2, dropout=cfg.ae_dropout)
            eval_model.set_weights(ms_artifacts["ms_model"].get_weights())

            X_fc_ft = X_budget[:, :context_steps, :]
            Y_fc_ft = X_budget[:, -1, :][:, score_idx]
            eval_model, _ = train_model_fixed_epochs(
                eval_model, X_fc_ft, Y_fc_ft,
                epochs=ft_epochs, lr=ft_lr,
                batch_size=batch_size_used, verbose=0)
            eval_model.save(ft_path)
            print(f"  [{target_city}] FT-Modell gespeichert: {ft_path}")
        train_time = time.time() - t0

    # ── Scoring ──────────────────────────────────────────────
    X_fc_input    = X_all[:, :context_steps, :]
    X_actual_last = X_all[:, -1, :]
    meta["score_fc_raw"] = compute_forecast_scores(
        eval_model, X_fc_input, X_actual_last,
        score_features, ae_features, mode="directional")

    # ── EVAL A: Strict (MS-Threshold) ────────────────────────
    result_strict = evaluate_fixed_threshold(
        meta, "score_fc_raw", ms_artifacts["ms_threshold_raw"],
        f"MS_{mode}_{target_city}_Strict",
        target_data["train_end"], target_data["val_end"])

    # ── EVAL B: Fair (Target-Val Z-Norm) ─────────────────────
    meta_znorm = znorm_score_by_hour_station(
        meta, "score_fc_raw",
        target_data["train_end"], target_data["val_end"],
        new_col="score_fc_znorm_hs")
    result_fair = evaluate_v17b(
        meta_znorm, "score_fc_znorm_hs",
        f"MS_{mode}_{target_city}_Fair",
        val_start=target_data["train_end"], test_start=target_data["val_end"])

    # ── Raw PR-AUC ───────────────────────────────────────────
    test_m = meta[meta["split_eval"] == "test"]
    y_test = test_m["synth_label"].astype(int).values
    raw_pr_auc = None
    if len(np.unique(y_test)) > 1:
        raw_pr_auc = float(average_precision_score(y_test, test_m["score_fc_raw"].values))

    combined = {
        "target_city": target_city,
        "source_cities": ",".join(ms_artifacts["source_cities"]),
        "n_sources": len(ms_artifacts["source_cities"]),
        "budget_days": budget_days,
        "mode": mode,
        "n_ft_sequences": n_ft,
        "actual_ft_days": round(actual_days, 2) if budget_days > 0 else 0,
        "train_time_sec": round(train_time, 2),
        "strict_test_pr_auc_raw": result_strict.get("test_pr_auc_raw"),
        "strict_test_f1": result_strict.get("test_f1"),
        "strict_test_prec": result_strict.get("test_prec"),
        "strict_test_recall": result_strict.get("test_recall"),
        "strict_pred_pos_rate": result_strict.get("pred_pos_rate_test"),
        "fair_test_pr_auc": result_fair.get("test_pr_auc"),
        "fair_test_f1": result_fair.get("test_f1"),
        "fair_test_prec": result_fair.get("test_prec"),
        "fair_test_recall": result_fair.get("test_recall"),
        "test_pr_auc_raw": raw_pr_auc,
        "n_test": result_strict.get("n_test"),
        "n_test_anom": result_strict.get("n_test_anom"),
        "ms_threshold": ms_artifacts["ms_threshold_raw"],
    }
    return combined, meta


---
## Exp4-Loop

Pro Target-Stadt:
1. Multi-Source-Modell einmalig trainieren (oder laden)
2. Zero-Shot (budget=0) evaluieren
3. Fine-Tune mit verschiedenen Budgets

In [28]:
# ══════════════════════════════════════════════════════════════
# Exp4-2 — Multi-Source Loop
# ══════════════════════════════════════════════════════════════
import gc

pred_dir    = f"{RESULTS_DIR}/predictions"
summary_dir = f"{RESULTS_DIR}/summary"
for d in [pred_dir, summary_dir]:
    os.makedirs(d, exist_ok=True)

checkpoint_path = f"{summary_dir}/exp04_multi_source_checkpoint.csv"
all_summary_rows = []
done_runs = set()

if os.path.exists(checkpoint_path):
    ckpt_df = pd.read_csv(checkpoint_path)
    all_summary_rows = ckpt_df.to_dict(orient="records")
    if {"target_city", "budget_days"}.issubset(ckpt_df.columns):
        done_runs = set(zip(
            ckpt_df["target_city"].astype(str),
            ckpt_df["budget_days"].astype(int)))
    print(f"Checkpoint geladen: {len(done_runs)} Runs abgeschlossen.")
else:
    print("Kein Checkpoint. Starte frisch.")

ms_model_cache = {}

for target_city, conf in MULTI_SOURCE_CONFIGS.items():
    source_cities = conf["sources"]
    source_key = "__".join(sorted([city_to_key(c) for c in source_cities]))

    print("\n" + "\u2588"*100)
    print(f"\u2588  TARGET: {target_city} | Sources: {', '.join(source_cities)}")
    print("\u2588"*100)

    # ── Multi-Source-Modell trainieren/laden ──────────────────
    if source_key not in ms_model_cache:
        keras.backend.clear_session()
        ms_model_cache[source_key] = build_multi_source_model(
            source_cities, cfg, MODELS_DIR, MS_EPOCHS, MS_LR)

    ms_artifacts = ms_model_cache[source_key]

    # ── Zero-Shot + Fine-Tune Budgets ────────────────────────
    for budget_days in FT_BUDGETS_DAYS:
        run_key = (target_city, budget_days)
        if run_key in done_runs:
            mode = "ZeroShot" if budget_days == 0 else f"FT_{budget_days}d"
            print(f"\n  [{target_city}] {mode} bereits im Checkpoint \u2014 \u00fcbersprungen.")
            continue

        keras.backend.clear_session()

        try:
            result, meta = run_multi_source_transfer(
                ms_artifacts, target_city, cfg, budget_days,
                FT_EPOCHS, FT_LR, MODELS_DIR,
                seed_offset=hash(target_city) % 100)

            if result is None:
                continue

            # Window-Scores
            mode = "ZeroShot" if budget_days == 0 else f"FT_{budget_days}d"
            model_name = f"fc_ms_{mode}_{source_key}"
            save_window_level_scores(
                meta, target_city, model_name, pred_dir,
                score_cols=["score_fc_raw"],
                extra_cols=["station_activity_raw", "station_activity_log1p",
                            "station_activity_group"])

            all_summary_rows.append(result)
            pd.DataFrame(all_summary_rows).to_csv(checkpoint_path, index=False)
            print(f"  Checkpoint aktualisiert.")

            del meta
            gc.collect()

        except Exception as e:
            print(f"  FEHLER: {e}")
            pd.DataFrame(all_summary_rows).to_csv(checkpoint_path, index=False)
            raise

print("\n" + "\u2588"*100)
print("\u2588  Exp4 \u2014 Multi-Source ABGESCHLOSSEN")
print("\u2588"*100)


Checkpoint geladen: 12 Runs abgeschlossen.

████████████████████████████████████████████████████████████████████████████████████████████████████
█  TARGET: gießen | Sources: Offenburg, leverkusen, marburg
████████████████████████████████████████████████████████████████████████████████████████████████████
[MULTI-SOURCE] Training auf: Offenburg, leverkusen, marburg

  [1/3] Lade Offenburg...

  DATEN-PIPELINE: Offenburg
  Roh: 143,369 Zeilen, 28 Stationen
  Zeitraum: 2023-07-20 bis 2026-02-02

  [1] Aggregation...
      Shape: (89574, 22)
  [2] Gap-Fill...
      89,574 -> 520,689 (+431,115)
  [3] Stationen, Regime, Splits...
  Aktive Stationen (Offenburg): 16
  Regime: {'mid': 6, 'high': 5, 'low': 5}
  Zeitraum: 2023-07-20 bis 2026-02-02 (929 Tage)
  Split: Train < 2025-04-02, Val < 2025-08-19, Test ab 2025-08-19
  [4] Statistische Scoring-Pipeline...
  Injection: requested=1763, injected=1763
    point_spike: req=1763, inj=1763
    Offenburg: 232,017 Train-Zeilen

  [2/3] Lade leverkuse

In [29]:
# ══════════════════════════════════════════════════════════════
# Exp4-3 — Summary
# ══════════════════════════════════════════════════════════════
summary_df = pd.DataFrame(all_summary_rows)
summary_path = f"{summary_dir}/exp04_multi_source_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Summary gespeichert: {summary_path}")

if len(summary_df) == 0:
    print("Keine Ergebnisse.")
else:
    print(f"\n{'='*120}")
    print("MULTI-SOURCE ERGEBNISSE")
    print(f"{'='*120}")

    display_cols = ["target_city", "source_cities", "mode", "budget_days",
                    "n_ft_sequences", "test_pr_auc_raw",
                    "strict_test_f1", "strict_test_prec", "strict_test_recall",
                    "fair_test_pr_auc", "fair_test_f1"]
    display_cols = [c for c in display_cols if c in summary_df.columns]
    print(summary_df[display_cols].sort_values(
        ["target_city", "budget_days"]).to_string(index=False))

    # ── Pro-Target-Vergleich ─────────────────────────────────
    print(f"\n{'='*120}")
    print("PRO TARGET: Multi-Source Progression")
    print(f"{'='*120}")
    for target in sorted(summary_df["target_city"].unique()):
        tgt_df = summary_df[summary_df["target_city"] == target].sort_values("budget_days")
        print(f"\n  {target} (Sources: {tgt_df['source_cities'].iloc[0]}):")
        for _, row in tgt_df.iterrows():
            mode = row.get("mode", "?")
            raw = row.get("test_pr_auc_raw", None)
            s_f1 = row.get("strict_test_f1", None)
            f_f1 = row.get("fair_test_f1", None)
            n_seq = row.get("n_ft_sequences", 0)
            line = f"    {mode:>12s}: raw PR-AUC={raw:.4f}" if raw else f"    {mode:>12s}: raw=N/A"
            if s_f1 is not None: line += f"  strict-F1={s_f1:.4f}"
            if f_f1 is not None: line += f"  fair-F1={f_f1:.4f}"
            if n_seq > 0: line += f"  [{n_seq} seq]"
            print(line)


Summary gespeichert: /content/drive/MyDrive/BA_Colab/data/exp04_multi_source/summary/exp04_multi_source_summary.csv

MULTI-SOURCE ERGEBNISSE
 target_city                          source_cities     mode  budget_days  n_ft_sequences  test_pr_auc_raw  strict_test_f1  strict_test_prec  strict_test_recall  fair_test_pr_auc  fair_test_f1
      bochum                         kassel,marburg ZeroShot            0               0         0.688993        0.653349          0.774373            0.565041          0.491399      0.594436
      bochum                         kassel,marburg    FT_7d            7            9399         0.695117        0.667054          0.778833            0.583333          0.502692      0.603050
      bochum                         kassel,marburg   FT_14d           14           18914         0.694573        0.659649          0.776860            0.573171          0.486772      0.600097
      bochum                         kassel,marburg   FT_30d           30           409